In [1]:
import os
os.environ["PYTHONNET_RUNTIME"] = "coreclr"
os.environ["DOTNET_SYSTEM_DRAWING_USE_GDIPLUS"] = "1"

import clr
import numpy as np
import time
from datetime import timedelta
from pythonnet import load
from System import Environment, Array, String

# Load .NET Core runtime
load("coreclr")

# Try to import System.IO; fallback to os.chdir if it fails
try:
    from System.IO import Directory, Path, File
    use_dotnet_dir = True
except Exception as e:
    print(f"Note: System.IO import failed ({e}), using os.chdir instead")
    import os
    use_dotnet_dir = False

# Define DWSIM path
dwsimpath = "/usr/local/lib/dwsim/"

# Set working directory
if use_dotnet_dir:
    Directory.SetCurrentDirectory(dwsimpath)
else:
    os.chdir(dwsimpath)

# Add DWSIM assemblies
clr.AddReference(dwsimpath + "CapeOpen.dll")
clr.AddReference(dwsimpath + "DWSIM.Automation.dll")
clr.AddReference(dwsimpath + "DWSIM.Interfaces.dll")
clr.AddReference(dwsimpath + "DWSIM.GlobalSettings.dll")
clr.AddReference(dwsimpath + "DWSIM.SharedClasses.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.dll")
clr.AddReference(dwsimpath + "DWSIM.UnitOperations.dll")
clr.AddReference(dwsimpath + "DWSIM.Inspector.dll")
clr.AddReference(dwsimpath + "System.Buffers.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.ThermoC.dll")

# Now import DWSIM types
from DWSIM.Interfaces.Enums.GraphicObjects import ObjectType
from DWSIM.Thermodynamics import Streams, PropertyPackages
from DWSIM.UnitOperations import UnitOperations
from DWSIM.Automation import Automation3
from DWSIM.GlobalSettings import Settings

print("DWSIM imports successful!")

DWSIM imports successful!


In [2]:
# Create an instance of the Automation3 class from the DWSIM.Automation module
# This class provides methods for automating tasks in DWSIM, such as creating and manipulating flowsheets
interf = Automation3()

In [3]:
# Creates a flowsheet
sim = interf.CreateFlowsheet()

In [4]:
# Add Compounds
sim.AddCompound("Methane")
sim.AddCompound("Water")
sim.AddCompound("Hydrogen")
sim.AddCompound("Carbon monoxide")
sim.AddCompound("Carbon dioxide")

In [5]:
feed = sim.AddFlowsheetObject('Material Stream','feed')
product = sim.AddFlowsheetObject('Material Stream','product')
energy_stream = sim.AddFlowsheetObject('Energy Stream','energy_stream')
pfr_01 = sim.AddFlowsheetObject('Plug-Flow Reactor (PFR)','pfr_01')

In [6]:
feed = feed.GetAsObject()
product = product.GetAsObject()
energy_stream = energy_stream.GetAsObject()
pfr_01 = pfr_01.GetAsObject()

In [7]:
sim.ConnectObjects(feed.GraphicObject, pfr_01.GraphicObject, -1, -1)
sim.ConnectObjects(pfr_01.GraphicObject, product.GraphicObject, -1, -1)
sim.ConnectObjects(energy_stream.GraphicObject, pfr_01.GraphicObject, -1, -1)

In [8]:
sim.AutoLayout()

In [9]:
feed.SetOverallComposition(Array[float]([0.4975, 0.4975, 0.005, 0.0, 0.0])) # Feed composition (50% CO and 50% H2O)
feed.SetTemperature(973.15) # K
feed.SetPressure(101325.0) # Pa
feed.SetMassFlow(0.0833) # kg/s

'feed: mass flow set to 0.0833 kg/s'

In [10]:
# property package
Thermo_Package_UNIQUAC = sim.CreateAndAddPropertyPackage("Peng-Robinson (PR)")

In [11]:
# Creating the conversion reaction for CH4 combustion
from System.Collections.Generic import Dictionary
from System import String, Double

# ------------------------------------------------------------
# 2. Define stoichiometry and kinetic orders
# ------------------------------------------------------------
from System.Collections.Generic import Dictionary
from System import String, Double

# Stoichiometry: CH4 + H2O -> CO + 3H2
stoich = Dictionary[String, Double]()
stoich.Add("Methane", -1.0)
stoich.Add("Water", -1.0)
stoich.Add("Hydrogen", 3.0)
stoich.Add("Carbon monoxide", 1.0)

# Base compound (required – usually the main reactant)
base_compound = "Methane"

# Reaction phase – vapor phase steam reforming
reaction_phase = "Vapor"

# Basis – "Partial Pressure" means rate depends on partial pressures (Pa)
basis = "Partial Pressure"

# Amount units – must match the basis (Pa for partial pressure)
amount_units = "Pa"

# Rate units – typical for heterogeneous catalytic: mol per kg catalyst per second
rate_units = "mol/kg h"


# Build expressions as strings (use 'T' for temperature in Kelvin)
numerator = f"4.22E+15*exp(-240100/(8.314*T))/(P1^2.5)*((R1*R2)-(P1^3*P2)/(exp(30.42-27106/T)))"

denominator = f"(1+6.65E-4*exp(38280/(8.314*T))*R1+8.23E-5*exp(70650/(8.314*T))*P2+6.12E-9*exp(82900/(8.314*T))*P1+1.77E+5*exp(-88680/(8.314*T))*R2/P1)^2"

# -----------------------------------------------------------------
# 3. Create the heterogeneous catalytic reaction
# -----------------------------------------------------------------
hetcat_reaction = sim.CreateHetCatReaction(
    "MethaneSteamReforming",                 # name
    "CH4 + H2O -> CO + 3H2 (LH kinetics)",   # description
    stoich,                                    # stoichiometry
    base_compound,                             # base compound
    reaction_phase,                            # phase
    basis,                                      # basis
    amount_units,                               # amount units
    rate_units,                                 # rate units
    numerator,                                  # numerator expression
    denominator                                 # denominator expression
)

# Add reaction to flowsheet
sim.AddReaction(hetcat_reaction)

# ------------------------------------------------------------
# 4. Create a reaction set and add the reaction
# ------------------------------------------------------------
reaction_set = sim.CreateReactionSet("Reforming Set", "Methane steam reforming")
sim.AddReactionSet(reaction_set)
sim.AddReactionToSet(hetcat_reaction.Name, reaction_set.Name, True, 0)

# Setting reaction set to the conversion reactor
from DWSIM.UnitOperations.Reactors import OperationMode
pfr_01.ReactorOperationMode = OperationMode.Isothermic
pfr_01.ReactionSetID = reaction_set.Name
pfr_01.ReactionSet = reaction_set
pfr_01.Volume = 1 # m³
pfr_01.Length = 1 # m
pfr_01.CatalystLoading = 0.366 # kg/m³
pfr_01.CatalystParticleDiameter = 0.002 # m
pfr_01.CatalystVoidFraction = 0.4

In [12]:
# request a calculation
errors = interf.CalculateFlowsheet4(sim)
list(errors)

[]

In [13]:
# save file

fileNameToSave = Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.Desktop), "/workspace/00 FlowSheet Automation/15 PFR Automation/15 PFR Automation.dwxmz")

interf.SaveFlowsheet(sim, fileNameToSave, True)